# PPO: train predators

In [ ]:
import jax
from flock.env import EnvConfig, RandomPolicy, run_episode
from flock.env.rules import PredatorPrey
from flock.env.viz import animate, plot_stats
from flock.train.naive import FleePredators
from flock.train.ppo import make_policy, train

In [ ]:
rules = PredatorPrey(
    n_predators=5,
    n_prey=20,
    catch_radius=0.3,
    k_teammates=4,
    k_opponents=5,
)
env_cfg = EnvConfig(
    max_steps=400,
    dt=0.1,
)

key = jax.random.key(42)
policy = make_policy(rules.teams[0].k_teammates, rules.teams[0].k_opponents, key=key)
prey_policy = FleePredators(rules.teams[1])

### Before training

In [ ]:
%matplotlib widget
sim_before = run_episode(env_cfg, rules, jax.random.key(0), (policy, prey_policy))
_ = animate(sim_before)

In [ ]:
_ = plot_stats(sim_before)

### Train

In [ ]:
trained_policy = train(
    env_cfg, rules, policy, prey_policy,
    key=jax.random.key(1),
    n_iters=256,
    n_arenas=128,
    n_epochs=4,
)

### After training

In [ ]:
sim_after = run_episode(env_cfg, rules, jax.random.key(0), (trained_policy, prey_policy))
_ = animate(sim_after)

In [ ]:
_ = plot_stats(sim_after)

In [ ]:
from flock.env.serialize import save, load
save(sim_after, "ppo2")
anim = animate(sim_after)
anim.save("ppo2.mp4", fps=30, extra_args=["-vcodec", "libx264"], bitrate=-1)